# 03 · The agent loop

An agent is a loop. We send the conversation *and the tool schemas* to the model. If the model answers with
a request to call tools, we run them, append the results to the conversation and send everything again.
If it answers with text, that text is the final answer. That is all there is to it; the rest of this
notebook is about the details that make the loop work.

The tools from notebook 02 now live in the project package, `src/aueb_agents_ca/`, so they can be imported
here and later by the API. `uv sync` installed the package into the virtual environment, which is why
`import aueb_agents_ca` works from anywhere.

In [7]:
import json
from typing import Any

import httpx
from ddgs import DDGS
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage, ChatCompletionMessageParam

from aueb_agents_ca.settings import Settings
from aueb_agents_ca.tools import ReviewTools, TOOL_SCHEMAS

settings: Settings = Settings.from_env()  # reads .env: provider URL, key, model names
# Explicit limits: the SDK's defaults are a 10-minute timeout and 2 retries, which turns a stalled
# request into a stalled notebook. Retries cover the 429 (rate limit) and 503 (overloaded) replies.
llm: OpenAI = OpenAI(base_url=settings.base_url, api_key=settings.api_key, timeout=60, max_retries=3)

# The tools receive everything they depend on from the outside.
tools: ReviewTools = ReviewTools(
    llm=llm,
    settings=settings,
    http=httpx.Client(follow_redirects=True, timeout=10, headers={"User-Agent": "Mozilla/5.0"}),
    search=DDGS(),
)
# aueb_agents_ca.tools.build_tools(settings) does this wiring in one line; it is spelled out here on purpose.
print("model:", settings.model, "| tools:", list(tools.registry))

model: gemini-3.5-flash-lite | tools: ['search_reviews', 'analyze_pages']


## 1. Why the tools are built this way

In notebook 02 the tools were loose functions that used a `client` created at the top of the notebook. In
the package they are methods of `ReviewTools`, and the class is handed its dependencies (the LLM client,
the HTTP client, the search engine, the settings) when it is created. This is *dependency injection*, and
it buys three things:

* **No surprises at import time.** Importing the module creates no clients and reads no `.env`; the caller
  decides when and how.
* **One code path for every provider.** Gemini, Groq or Ollama is just a different `settings`.
* **Tests without the internet.** `tests/test_tools.py` passes a fake LLM, an `httpx` mock transport that
  answers `403`, and a fake search engine, and checks the tools' behaviour in under a second. Run
  `uv run pytest` to see it.

The package version is also hardened in ways the lesson in notebook 02 skipped: search failures come back
as text instead of exceptions, embedding requests are batched and capped at the 40 most frequent aspect
names, pages with no readable text are reported as failed, and problems are logged rather than printed.
Read `tools.py` side by side with notebook 02 to see what production-readiness adds.

`tools.registry` maps each tool's name to the method, which is what the loop needs to execute a call.

## 2. What the model sees

The model never sees Python. For each tool it receives a JSON schema: the name, a description, and the
parameters with their types. `TOOL_SCHEMAS` in `tools.py` holds one per tool. The description is the
only thing the model has when deciding whether and when to call a tool, so it carries the usage advice.

In [ ]:
print(json.dumps(TOOL_SCHEMAS[1], indent=2))

Notice that `analyze_pages` takes **URLs**, not texts. The tool downloads the pages itself and keeps them
(`tools.pages`); the model only ever handles URLs. The alternative, a `fetch_page` tool whose output the
model passes on to the analysis, would mean the model generating tens of thousands of characters of tool
arguments, slowly and unreliably. Give the model *handles*, never make it echo *payloads*.

**Why only two tools.** `ReviewTools.fetch_page` exists, but it is not in the schemas: `analyze_pages` calls it
internally. An earlier version exposed it to the model, and a small model given the choice would sometimes
search once, read three pages and write the summary from the raw text, skipping the analysis entirely.
Every tool you expose is a decision you hand to the model; expose only what the task needs, and the
agent's behaviour becomes far more predictable.

## 3. The loop

![The agent loop](img/agent_loop.svg)

Four details matter:

1. **One iteration is one model call**, with the whole conversation and the tool schemas attached.
2. **The exit condition is "no tool calls"**, not a fixed number of steps: the model decides how many
   steps it needs. `max_steps` is only a safety net against a model that never stops asking.
3. **Tool results go back as `tool` messages**, each tied to the id of the call it answers. The model
   can request several tools in one turn, so the loop runs all of them before calling the model again.
4. **The model's message is appended as the object we received**, not rebuilt from its text. Some
   providers (Gemini among them) attach extra fields to it that must be returned on the next turn.

The system prompt is where we tell the model what a good run looks like, and where the agent's **scope**
is enforced: a request that is not about product reviews gets a "not supported" answer. That check
belongs here and not inside a tool: tools do exactly what their name says and never second-guess their
input, while the agent is the one that understands the task.

In [9]:
SYSTEM_PROMPT = """You are a customer-analytics assistant. Given a product, find its reviews online and write a
short summary for a marketing team: overall sentiment, the aspects people praise, the aspects they complain
about, each backed by a quote, and the list of sources.
Plan: run search_reviews two or three times with different wording (for example the product name alone,
'long term review', and 'vs' a rival) to collect as many distinct review pages as possible; then call
analyze_pages once with every review page found, skipping shops, adverts and duplicates; then write the
summary from the returned table. Never invent reviews or quotes.
If the task is not about summarising the reviews of a product, reply that it is not supported and stop."""


# The conversation mixes two kinds of entries: the dicts we write (system, user, tool messages) and the
# message objects the model returns, which we append as they are. The alias makes that explicit.
Conversation = list[ChatCompletionMessageParam | ChatCompletionMessage]


def run_agent(
    task: str, llm: OpenAI, tools: ReviewTools, model: str, max_steps: int = 8
) -> tuple[str, Conversation]:
    """Run the agent loop on one task. Returns the final answer and the full conversation."""
    messages: Conversation = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]

    for step in range(max_steps):
        response = llm.chat.completions.create(model=model, messages=messages, tools=TOOL_SCHEMAS)
        message = response.choices[0].message
        messages.append(message)  # the object itself, see detail 4 above
        print(f"step {step}: finish_reason={response.choices[0].finish_reason}, "
              f"prompt_tokens={response.usage.prompt_tokens}")

        if not message.tool_calls:  # no tool requested: this is the final answer
            return message.content, messages

        for call in message.tool_calls:  # run every requested tool and report back
            args: dict[str, Any] = json.loads(call.function.arguments)
            print(f"   -> {call.function.name}({json.dumps(args)[:100]})")
            result: Any = tools.registry[call.function.name](**args)  # whatever the tool returns, JSON-serialisable
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

    return "Stopped: too many steps without a final answer.", messages

In [10]:
examples: list[str] = [
    "Summarise the reviews of the Sony WH-1000XM5 headphones.",
    "Dungeon Crawler Carl Review.",
    "Dungeon Crawler Carl Book 1 summary.",
    "What's the weather in Athens"
]

# select which prompt to run
prompt = examples
answer, messages = run_agent("Summarise the reviews of the Sony WH-1000XM5 headphones.", llm, tools, settings.model)
# answer: str, messages: Conversation
print()
print(answer)

step 0: finish_reason=tool_calls, prompt_tokens=431
   -> search_reviews({"query": "Sony WH 1000XM5 reviews"})
step 1: finish_reason=tool_calls, prompt_tokens=3432
   -> fetch_page({"url": "https://www.whathifi.com/reviews/sony-wh-1000xm5"})
   -> fetch_page({"url": "https://www.soundguys.com/sony-wh-1000xm5-review-71783/"})
   -> fetch_page({"url": "https://www.tomsguide.com/reviews/sony-wh-1000xm5"})
step 2: finish_reason=stop, prompt_tokens=17713

Here is a customer-analytics summary of the Sony WH-1000XM5 wireless headphones, based on reviews from leading tech publications.

### **Overall Sentiment**
The sentiment surrounding the Sony WH-1000XM5 is **overwhelmingly positive**. Reviewers widely praise them as class-leading or near-top-tier active noise-canceling (ANC) headphones, noting significant generational improvements in sound clarity, noise isolation, and call quality compared to their predecessors. 

---

### **Aspects People Praise**
* **Active Noise Cancellation (ANC):** R

## 4. Reading the trace

* `prompt_tokens` grows at every step because the whole conversation is resent. Search results add a
  few thousand tokens; the aspect table adds a couple of thousand more. Page texts never enter the
  conversation: each one would add about 5 000 tokens to every later call.
* The plan in the system prompt is advice, not a program: the model may search once or three times, and
  a small model sometimes deviates. What it cannot do is skip the analysis, because analysing is the only
  way it has to use a page.
* **Rate limits bound everything else.** On the Gemini free tier `gemini-3.5-flash-lite` allows 15 requests
  per minute and 500 per day, and one run of this agent costs about 20 requests: two searches, one
  extraction call per page, the loop's own calls. The embedding model allows 100 requests per minute and
  1 000 per day, and *every text in a batch counts as a request*, which is why `merge_aspects` embeds only
  the 40 most frequent aspect names. `gemini-3.8-flash` allows 20 requests a day: unusable for an agent,
  which is why it is not the default. The client's `max_retries=3` turns a 429 into a short wait rather
  than a crash. Budget your experiments: a key is good for roughly 20 runs a day.
* How many pages get analysed is bounded by the search, not by the prompt. The search backend returns
  about ten results per query whatever `max_results` says, and only some of them are reviews; the rest are
  shops, adverts or pages that refuse scripted downloads. That is why the prompt asks for several
  differently worded searches: each one adds a few new pages. The model reads the tool descriptions as
  carefully as the system prompt, so a limit stated in a schema (an earlier version said "3 to 5 URLs")
  overrides a prompt that asks for more.
* Every quote in the summary came from the aspect table, which came from the pages. The summary is
  grounded, and you can check any claim against `tools.pages`.

The conversation is just a list. Here is its shape:

In [11]:
for m in messages:
    if isinstance(m, dict):  # our own messages: system, user, tool
        print(f"{m['role']:>9} | {m['content'][:90]!r}")
    elif m.tool_calls:  # the model asking for tools
        print(f"{'assistant':>9} | tool_calls: {[c.function.name for c in m.tool_calls]}")
    else:  # the model's final answer
        print(f"{'assistant':>9} | {m.content[:90]!r}")

   system | 'You are a customer-analytics assistant. Given a product, find its reviews online and write'
     user | 'Summarise the reviews of the Sony WH-1000XM5 headphones.'
assistant | tool_calls: ['search_reviews']
     tool | '[{"title": "Sony wh 1000xm5 review - Best Brands Side by Side", "url": "https://www.bing.c'
assistant | tool_calls: ['fetch_page', 'fetch_page', 'fetch_page']
     tool | '"What Hi-Fi? Verdict\\nIt can be tricky for a manufacturer to push the sound performance of'
     tool | '"All products featured are independently chosen by us. However, SoundGuys may receive a co'
     tool | '"Tom\'s Guide Verdict\\nSony demonstrates across-the-board expertise in the XM5 with a great'
assistant | 'Here is a customer-analytics summary of the Sony WH-1000XM5 wireless headphones, based on '


### The whole conversation, and what the model was "thinking"

The list above is everything the model ever had: no hidden memory, no state on the server between
calls. Printing it in full shows exactly what each decision was based on. Tool outputs are trimmed here
because the search results and the aspect table run to thousands of characters.

What about the model's reasoning? Gemini models do think before they answer (the encrypted
`thought_signature` attached to each message is that thinking, which is why we send the message object
back unchanged), but the OpenAI-compatible endpoint never returns the thoughts themselves. Groq's
gpt-oss models do: the text arrives as `message.reasoning`. The cell below prints it whenever the
provider exposes it. Switch `.env` to Groq, restart the kernel and run the notebook again to read it.
Even then, what you get is a summary the provider chose to show, not a transcript of the computation.

In [12]:
def show_conversation(messages: Conversation, max_chars: int = 1500) -> None:
    """Print every message in full (tool outputs trimmed), plus the model's reasoning when the provider returns it."""
    for i, m in enumerate(messages):
        if isinstance(m, dict):  # system, user and tool messages are plain dicts
            role, body = m["role"], m["content"]
        else:  # the model's messages: maybe reasoning, maybe tool calls, maybe text
            role = "assistant"
            reasoning = getattr(m, "reasoning", None) or getattr(m, "reasoning_content", None)
            if reasoning:
                print(f"\n[{i}] assistant (reasoning)\n{reasoning}")
            body = m.content or ""
            if m.tool_calls:
                body += "\n".join(f"-> {c.function.name}({c.function.arguments})" for c in m.tool_calls)
        if len(body) > max_chars:
            body = body[:max_chars] + f" ... [{len(body) - max_chars} more characters]"
        print(f"\n[{i}] {role}\n{body}")


show_conversation(messages)


[0] system
You are a customer-analytics assistant. Given a product, find its reviews online and write a
short summary for a marketing team: overall sentiment, the aspects people praise, the aspects they complain
about, each backed by a quote, and the list of sources.
Plan: run search_reviews two or three times with different wording (for example the product name alone,
'long term review', and 'vs' a rival) to collect as many distinct review pages as possible; then call
analyze_pages once with every review page found, skipping shops, adverts and duplicates; then write the
summary from the returned table. Never invent reviews or quotes.
If the task is not about summarising the reviews of a product, reply that it is not supported and stop.

[1] user
Summarise the reviews of the Sony WH-1000XM5 headphones.

[2] assistant
-> search_reviews({"query":"Sony WH 1000XM5 reviews"})

[3] tool
[{"title": "Sony wh 1000xm5 review - Best Brands Side by Side", "url": "https://www.bing.com/aclick?ld=e8

## 5. Where this goes next

The loop above is the whole agent, and it is about thirty lines. What a framework would add is
conveniences around it: automatic schema generation from type hints, retries, streaming, tracing.
Knowing the loop means knowing what those frameworks do.

Things that can go wrong, and that you should try: ask about a product with few reviews; pass a model
that is weaker at tool use (Ollama with a small model); remove the plan from the system prompt and watch
which tools the model picks. The tools report errors as text, so the loop survives; whether the model
recovers well is a question of model quality.

The package has the same loop in `src/aueb_agents_ca/agent.py` (`run_agent`, with logging instead of prints);
that is the version the API in Step 3 imports. The notebook keeps its own copy so the whole agent is
readable on one screen.